## Problem description

A bike sharing operator needs to know how many bikes will be in use at each hour to decide how many to keep at the stations and when to rebalance or service them. Too few bikes leave riders without service, and too many leave idle bikes and wasted logistics.

**Business question:** How many bikes will be rented in a given hour, given the calendar and weather conditions?

**Dataset:** Bike Sharing, hourly file (UCI #275). It has 17,379 hourly records from the Capital Bikeshare system in Washington, D.C., covering 2011 and 2012, with calendar, weather and rental counts.

**Target:** `cnt`, the total number of rentals in the hour. The result is reported in bikes per hour.

## What this notebook does

- **ETL:** load the hourly file, keep an untouched copy in `data/raw`, fix types, standardize names, check duplicates and missing values, remove `casual` and `registered` (they add up to `cnt` and would leak the target), encode hour and month as sine and cosine, and save the clean data with a data dictionary in `data/processed`. Only transformations that do not learn from the data are done here.
- **EDA:** profile the data, study the distribution of `cnt` and its relation to the other variables, compare Pearson and Spearman correlations, and test three hypotheses.
- **Data quality log:** record each problem found, the decision taken and, for at least one of them, the metric before and after the fix.
- **Modeling:** split the test set before any transformation, build the preprocessing inside a `Pipeline`, and compare a baseline, Linear Regression, Ridge or Lasso and a tree ensemble with cross validation. Test a log transform of the target.
- **Evaluation:** report MAE, RMSE and R² on the test set, with a bootstrap confidence interval and a paired test between the two best models.
- **Interpretation:** coefficients and permutation importance, analysis of the 10 worst predictions, limitations and conclusions in business terms.

## 1. Setup

### 1.1 Imports

Libraries used in the notebook.

**Note**: New imports go in this block as they are needed. (remove this line after finished)

In [1]:
import io
import zipfile
from datetime import date
from pathlib import Path
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd

### 1.2 Constants

All project constants (seed, test size, source URL, file name and paths) are defined here. Paths are relative to the repository root, so the notebook runs from `notebooks/` and from Colab.

**Note**: New constants go in this block, not in later cells. (remove this line after finished)

In [2]:
# Reproducibility
SEED = 42
np.random.seed(SEED)

# Data split
TEST_SIZE = 0.2

# Data source
SOURCE_URL = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"
RAW_FILENAME = "hour.csv"

# Download
REQUEST_TIMEOUT_SECONDS = 60
USER_AGENT = "Mozilla/5.0"
BYTES_PER_MB = 1_000_000

# Paths, relative to the repository root (works from notebooks/ and from Colab)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = ROOT/"data"/"raw"/"bike_sharing"
PROCESSED_DIR = ROOT/"data"/"processed"/"bike_sharing"

## 2. ETL

### 2.1 Extraction

The UCI zip is downloaded on every run and only `hour.csv` is written to `data/raw/bike_sharing`, replacing any previous copy. The function prints the source, access date and size, which are recorded in the report.

In [3]:
# Downloads the zip file and returns its bytes
def fetch_zip(url, timeout=REQUEST_TIMEOUT_SECONDS, user_agent=USER_AGENT):
    request = Request(url, headers={"User-Agent": user_agent})
    with urlopen(request, timeout=timeout) as response:
        return response.read()

# Returns the bytes of a file inside the zip
def extract_member(payload, filename):
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        member = next((name for name in archive.namelist() if name.endswith(filename)), None)
        if member is None:
            raise FileNotFoundError(f"{filename} not found in the zip")
        return archive.read(member)

# Writes the bytes to disk, creating the folders and overwriting any previous copy
def save_raw(content, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(content)

# Prints the source, access date and size, which are recorded in the report
def log_download(url, path, size):
    print(f"Source:   {url}")
    print(f"Accessed: {date.today()}")
    print(f"Saved:    {path} ({size/BYTES_PER_MB:.2f} MB)")

# Downloads the dataset zip and saves only the raw file in raw_dir
def download_raw(url=SOURCE_URL, raw_dir=RAW_DIR, filename=RAW_FILENAME):
    payload = fetch_zip(url)
    content = extract_member(payload, filename)
    path = raw_dir/filename
    save_raw(content, path)
    log_download(url, path, len(content))
    return path

In [4]:
# Download the all raw data, and save specific file in the format .csv, also showing the logs
raw_path = download_raw()

Source:   https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip
Accessed: 2026-11-24
Saved:    /workspaces/MachineLearning-2026.2/data/raw/bike_sharing/hour.csv (1.16 MB)


### 2.2 Raw data overview

The raw file is loaded without any change to check its shape and first rows.

In [ ]:
# Read the file raw data hour.csv (as df)
df_raw = pd.read_csv(raw_path)
print("Raw data size:", df_raw.shape)
# Showing the firt elements
df_raw.head()

Raw data size: (17379, 17)


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


**Variables according to the source ([UCI Bike Sharing](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset)):**

| Variable   | Role    | Type        | Description                                                      | Units | Missing values |
|------------|---------|-------------|------------------------------------------------------------------|-------|----------------|
| instant    | ID      | Integer     | Record index                                                     |       | no             |
| dteday     | Feature | Date        | Date                                                             |       | no             |
| season     | Feature | Categorical | 1: winter, 2: spring, 3: summer, 4: fall                         |       | no             |
| yr         | Feature | Categorical | Year (0: 2011, 1: 2012)                                          |       | no             |
| mnth       | Feature | Categorical | Month (1 to 12)                                                  |       | no             |
| hr         | Feature | Categorical | Hour (0 to 23)                                                   |       | no             |
| holiday    | Feature | Binary      | Whether the day is a holiday (DC holiday schedule)               |       | no             |
| weekday    | Feature | Categorical | Day of the week                                                  |       | no             |
| workingday | Feature | Binary      | 1 if the day is neither weekend nor holiday, otherwise 0         |       | no             |
| weathersit | Feature | Categorical | Weather situation, from 1 (clear) to 4 (heavy rain, snow or fog) |       | no             |
| temp       | Feature | Continuous  | Normalized temperature, t_min = -8, t_max = +39                  | C     | no             |
| atemp      | Feature | Continuous  | Normalized feeling temperature, t_min = -16, t_max = +50         | C     | no             |
| hum        | Feature | Continuous  | Normalized humidity, divided by 100 (max)                        |       | no             |
| windspeed  | Feature | Continuous  | Normalized wind speed, divided by 67 (max)                       |       | no             |
| casual     | Other   | Integer     | Count of casual users                                            |       | no             |
| registered | Other   | Integer     | Count of registered users                                        |       | no             |
| cnt        | Target  | Integer     | Count of total rental bikes, casual plus registered              |       | no             |

### 2.3 Raw data inspection

Types, missing values, duplicates and value ranges are checked before any change. Problems found here go to the data quality log.

In [9]:
# Getting types and nule ones
print("Getting types and nule ones:\n")
df_raw.info()
# Showing the total of nule and total duplicates rows separately
print("\nNull rows:", df_raw.isna().sum())
print("\nDuplicated rows:", df_raw.duplicated().sum())

Getting types and nule ones:

<class 'pandas.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 17 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   instant     17379 non-null  int64  
 1   dteday      17379 non-null  str    
 2   season      17379 non-null  int64  
 3   yr          17379 non-null  int64  
 4   mnth        17379 non-null  int64  
 5   hr          17379 non-null  int64  
 6   holiday     17379 non-null  int64  
 7   weekday     17379 non-null  int64  
 8   workingday  17379 non-null  int64  
 9   weathersit  17379 non-null  int64  
 10  temp        17379 non-null  float64
 11  atemp       17379 non-null  float64
 12  hum         17379 non-null  float64
 13  windspeed   17379 non-null  float64
 14  casual      17379 non-null  int64  
 15  registered  17379 non-null  int64  
 16  cnt         17379 non-null  int64  
dtypes: float64(4), int64(12), str(1)
memory usage: 2.3 MB

Null rows: instant       0
dteday

**Findings:** 17,379 rows and 17 columns, with no missing values (as stated on the UCI page) and no duplicated rows. 16 columns are numeric (12 integer and 4 float) and the only non numeric one is `dteday`, stored as text, which must be converted to a date type.

In [ ]:
# Summary statistics to check the value range of each numeric column
df_raw.describe().T

,count,mean,std,min,25%,50%,75%,max
instant,17379.0,8690.000000,5017.029500,1.00,4345.5000,8690.0000,13034.5000,17379.0000
season,17379.0,2.501640,1.106918,1.00,2.0000,3.0000,3.0000,4.0000
yr,17379.0,0.502561,0.500008,0.00,0.0000,1.0000,1.0000,1.0000
mnth,17379.0,6.537775,3.438776,1.00,4.0000,7.0000,10.0000,12.0000
hr,17379.0,11.546752,6.914405,0.00,6.0000,12.0000,18.0000,23.0000
holiday,17379.0,0.028770,0.167165,0.00,0.0000,0.0000,0.0000,1.0000
weekday,17379.0,3.003683,2.005771,0.00,1.0000,3.0000,5.0000,6.0000
workingday,17379.0,0.682721,0.465431,0.00,0.0000,1.0000,1.0000,1.0000
weathersit,17379.0,1.425283,0.639357,1.00,1.0000,1.0000,2.0000,4.0000
temp,17379.0,0.496987,0.192556,0.02,0.3400,0.5000,0.6600,1.0000


**Findings:** All columns are within their documented ranges, `cnt` is right skewed (mean 189, median 142) and the weather variables are already normalized between 0 and 1.
Minimum values of 0 in `hum`, `windspeed` and `atemp` need a check, since they may be missing values coded as zero.

**To do**: Check the hum windspeed and atemp minimum equals a 0.

### 2.4 Types and column names

The date column is stored as text, so it is converted to datetime to allow validation and date based features. The abbreviated source names (`dteday`, `mnth`, `hr`, `hum`, `atemp`, `weathersit`, `cnt`) are renamed to full names, so the notebook, charts and report are readable without the data dictionary. The raw DataFrame is kept unchanged and the result goes to a new one.